# **Intialization**

In [ ]:
# Essential Libraries
import os
from os import path
import glob
from pathlib import Path
import datetime
import math
import warnings

# Data Handling
import numpy as np
import pandas as pd
import scipy

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# General
import tqdm
import itertools
from itertools import chain

# Widgets
import ipywidgets as widgets
from IPython.display import display

# Special Libraries
import nupack
from nupack import *

print(nupack.__version__)

# **Model Parameters**

In [ ]:
# Physical Model
oligo = 'dna'
ensemble = 'stacking'

# Solution/Environment
sodium = 0.060
magnesium = 0.005

# Concentration and Complex Size
conc = 1e-7
max_complex_size = 2

# **Core Functions**

In [ ]:
def melting_curve(strand_a, strand_b, conc = 1e-6, temp_min = 0, temp_max = 100, temp_step = 1, sodium = 0.150, magnesium = 0.03, oligo = 'dna', ensemble = 'stacking', show_curve = False):
    """
    Generates melting curves for analysis of DNA strands. Assumes a binary, equimolar mixture. Iteratively calculates percentage of materials bound at each temperature for a 
    given set of parameters using nupack. Calulates melting temperature as point at whcih 50% of strands are bound in duplex, which is determined by linearly interpolating the
    linear domain of the melting curve. Melting curves are generally sigmoidal and have a linear domain. 

    Parameters: 
    -   strand_a: str
            Sequence of strand A.
    
    -   strand_b: str
            Sequence of strand B
    
    -   conc: float
            Concentration of both strands in micromolarity.
    
    -   temp_min: float or int
            Minimum temperature for calculation in degrees Celsius
    
    -   temp_max: float or int
            Maximum temperature for calculation in degrees Celsius.
    
    -   temp_step: float or int
            Step between points on melting curve in degrees Celsius.

    -   sodium: float
            Concentration of sodium ions for model in millimolarity.

    -   magnesium: flaot
            Concentration of magnesium ions for model in millimolarity.

    -   oligo: str
            Oligonucleotide of interest. 'dna' for DNA, 'rna' for RNA.
        
    -   ensemble: str
            Determines how base stacking energies are calculated. should be 'stacking'.

    -   show_curve: bool
            Displays melting curve if True.

    Returns: 
    -   vals: np.ndarray
            (T, 2) array of temperatures and fraction of materials bound.
    
    -   tm: float
            Melting temperature of duplex. 
    """
    # Create strand and complex objects
    species_a = Strand(strand_a, name = 'a')
    species_b = Strand(strand_b, name = 'b')
    c = Complex([species_a, species_b], name = 'Duplex')

    # Create tube for ensemble. max_size=0 so that only complex c is evaluated
    tube = Tube(strands = {species_a : conc, species_b : conc}, complexes = SetSpec(include=[c], max_size = max_complex_size), name = 'Tube')
    
    # Melting curve calculation
    vals = []
    for i in np.arange(start = temp_min, stop = temp_max + 1, step = temp_step):
        # Generate model for temperature step
        model = Model(ensemble = ensemble, material = oligo, celsius = i, sodium = sodium, magnesium = magnesium)

        # Tube analysis at temperature step i
        tube_results = tube_analysis(tubes = [tube], model = model, compute = ['pfunc'])
        conc_results = complex_concentrations(tube = tube, data = tube_results)

        # Extract concentration of target ssDNA using concentration of target duplex
        duplex_conc = conc_results[tube].complex_concentrations[c]
        vals.append([i, conc - duplex_conc])

    # Convert result list to array
    vals = np.vstack(vals)

    # Normalize all concentrations against starting concentration of ssDNA
    vals[:,1] = vals[:,1]/conc

    # Indirect sort of vals to yield 10 rows with %dsDNA values closest to 50
    idx = np.argsort(np.abs(vals[:,1] - 0.5))[:9]

    # Use linear fit to determine melting point
    m, b = np.polyfit(idx, vals[idx,1], 1)
    tm = (0.5-b)/m
    
    if not show_curve: return vals, tm

    # Generate Figure
    fig, ax = plt.subplots(figsize = (10,6))
    ax.plot(vals[:,0], vals[:,1])
    fig.suptitle(f'Melting Curve, $T_m$={tm:.2f}°C')
    ax.set_xlabel('Temperature (°C)')
    ax.set_ylabel('Fraction Bound')
    plt.tight_layout()
    plt.show()

    # Return list of melting temps
    return vals, tm

# **Calculate Metling Curves for Experimentally Tested Fuel Sequences**

In [ ]:
# Input sequence of DNA leg (5' -> 3')
dna_leg = 'TTTTTTTTTTTTTTTGGGTAATGATTTTTT'

# Library of fuel strands
fuel_library = {
    '0C' : 'GCACCCATCTCTCTCTTTTTAATCATTA',
    '1C' : 'GCACCCATCTCTCTCTTTTTAATCATTAC',
    '2C' : 'GCACCCATCTCTCTCTTTTTAATCATTACC',
    '3C' : 'GCACCCATCTCTCTCTTTTTAATCATTACCC'
}

# Initialize container for results
result_dict = {}

# Iterate over strands to calculate melting temp against DNA leg
for k, v in fuel_library.items():
    print(f'Results for {k}')
    a = v
    b = dna_leg

    curve, tm = melting_curve(strand_a = a, strand_b = b)

    result_dict[k] = [curve, tm]

# Generate Figure
i = 0

fig, ax = plt.subplots(figsize = (10,6))
for k, v in result_dict.items():
    ax.plot(v[0][:,0], v[0][:,1], label = f'{k} Fuel, $T_m$={v[1]:.2f}°C')
    ax.vlines(x = v[1], ymin = 0, ymax = 1.1, linestyle = '--', color = f'C{i}')
    i += 1
# fig.suptitle(f'Melting Curve Comparison')
ax.set_xlabel('Temperature (°C)', fontsize = 16)
ax.set_ylabel('Fraction Unbound', fontsize = 16)
ax.tick_params(axis = 'both', labelsize =14)
ax.legend()
plt.axhline(y=0.5, xmin = 0, xmax = 100, linestyle='--', color = 'k')
plt.tight_layout()
plt.show()



In [ ]:
# Input sequence of DNA leg (5' -> 3')
dna_leg = 'TTTTTTTTTTTTTTTGGGTAATGATTTTTT'

# Input sequence of 3C fuel strand (5' -> 3')
fuel_strand = 'GCACCCATCTCTCTCTTTTTAATCATTACCC'


# Initialize container for results
result_dict = {}

# Iteratively calculate melting curves for digested DNA
for i in tqdm.tqdm(np.arange(start = 0, stop = 9), desc = 'Calculating Melting Curves'):
    if i == 0: a_name = '3C Fuel'
    elif i == 1: a_name = '1 base digested'
    else: a_name = f'{i} bases digested'
    
    a = fuel_strand[:-i] if i > 0 else fuel_strand
    b = dna_leg
    curve, tm = melting_curve(strand_a = a, strand_b = b)

    result_dict[a_name] = [curve, tm]

# Generate Melting Curves Figure
fig, ax = plt.subplots(figsize = (10,6))
for k, v in result_dict.items():
    ax.plot(v[0][:,0], v[0][:,1], label = f'{k}, $T_m$={v[1]:.2f}°C')
# fig.suptitle(f'Melting Curve Comparison')
ax.set_xlabel('Temperature (°C)', fontsize = 16)
ax.set_ylabel('Fraction Unbound', fontsize = 16)
ax.legend()
ax.tick_params(axis = 'both', labelsize =14)
plt.axhline(y=0.5, xmin = 0, xmax = 100, linestyle='--', color = 'k')
plt.tight_layout()
plt.show()

# Generate Tm Figure
fig, ax = plt.subplots(figsize = (10,6))
melting_temps = []
for k, v in result_dict.items():
    melting_temps.append(v[1])
ax.plot(melting_temps)
plt.axhline(y = 22, xmin = 0, xmax = 15, linestyle = '--', color = 'k', label = 'Room Temperature')
ax.set_xlabel('Number of Bases Hydrolyzed', fontsize = 16)
ax.set_ylabel('Melting Temperature (°C)', fontsize = 16)
ax.tick_params(axis = 'both', labelsize =14)
plt.legend()
plt.tight_layout()
plt.show()

# **Evaluate Thermodynamic Stability of Scrambled Fuel Sequences**

In [ ]:
np.random.seed(42)

bases = ['G','G','G','T','A','A','T','G','A','T','T']

dna_pairings = {
    'A' : 'T',
    'C' : 'G',
    'G' : 'C',
    'T' : 'A', 
}

fixed_region_1_leg = 'TTTTTTTTTTTTTTT'
fixed_region_2_leg = 'TTTTT'
fixed_region_fuel = 'GCACCCATCTCTCTCTTTTT'


def scramble_strand(input_base_list):
    """
    Creates scrambled sequences to assess impact of sequence on change in melting temperature. Uses random selection without replacement so strand sequence is scrambled. 
    
    Parameters: 
    -   input_base_list: str
        String of bases that will be scrambled
        
    Returns: 
    -   new_leg_frag: str
            String of scrambled bases in complementary region of DNA leg.
    """
    new_leg_frag = input_base_list
    while new_leg_frag == input_base_list:
        new_leg_frag = list(np.random.choice(input_base_list, size = len(input_base_list), replace = False))
    return ''.join(new_leg_frag)

def generate_scrambled_exoIII_leg_sequence(input_base_list, fixed_region_1, fixed_region_2):
    """
    Creates DNA leg sequences with scrambled complementary cores (between poly-T spacer and poly-T region to prevent hydrolysis by ExoIII).
    
    Parameters: 
    -   input_base_list: str
            String of bases that will be scrambled
    
    -   fixed_region_1: str
            Poly-T spacer between motor chassis and complementary region.
               
    -   fixed_region_2: str
            Poly-T spacer between complementary region and end of strand.
    
    Internal Dependencies: 
    -   scramble_strand
       
    Returns: 
    -   comp_region: str
            Complementary region of new leg sequence. Used to create complementary fuel sequence. 
    
    -   New leg sequence: str
    """
    comp_region = scramble_strand(input_base_list=input_base_list)
    return comp_region,''.join([fixed_region_1, comp_region, fixed_region_2])

def generate_complementary_exoIII_fuel_sequence(complementary_region, fixed_region):
    """
    Creates DNA fuel sequences with scrambled complementary region.
    
    Parameters: 
    -   complementary_region: str
            String of bases for which an exact complement will be generated
    
    -   fixed_region: str
            Spacer and anchor oligo handle.
       
    Returns:    
    -   New fuel sequence: str
    """
    result = [dna_pairings[char] for char in reversed(complementary_region)]
    result = ''.join(result)
    return  ''.join([fixed_region, result])

def exoIII_scrambled_duplex_library(input_base_list, N=100):
    """
    Creates dictionary of leg-fuel pairs. Legs are keys.
    
    Parameters: 
    -   input_base_list: str
            String of input bases from complementary region.
    
    -   N: int
            Number of iterations. 
            
    Internal Dependencies: 
    -   generate_scrambled_exoIII_leg_sequence
    -   generate_scrambled_exoIII_fuel_sequence
    
    Returns: 
    -   strands_dict: dict
            Dictionary of leg-fuel pairs. 
    """
    strands_dict = {}
    while len(strands_dict) < N:
        comp_strand, leg = generate_scrambled_exoIII_leg_sequence(input_base_list, fixed_region_1=fixed_region_1_leg, fixed_region_2=fixed_region_2_leg)
        fuel = generate_complementary_exoIII_fuel_sequence(complementary_region=comp_strand, fixed_region=fixed_region_fuel)
        strands_dict[leg] = fuel
    return strands_dict

s = exoIII_scrambled_duplex_library(input_base_list=bases, N = 4000)



In [ ]:
# Input sequence of DNA leg (5' -> 3')
c3_dna_leg = 'TTTTTTTTTTTTTTTGGGTAATGATTTTTTT'

# Input sequence of 3C fuel strand (5' -> 3')
v4_fuel_strand = 'GCACCCATCTCTCTCTTTTTAATCATTACCC'


# Initialize container for results
result_arr = []

lim = 9

# Iteratively calculate melting curves for digested DNA
for k, v in tqdm.tqdm(s.items()):
    temps = np.full(lim, fill_value=np.nan)
    for i in np.arange(start = 0, stop = lim):
        a_name = f'{v}-{i}'
        a = v[:-i] if i > 0 else v
        b = k

        if i == 0: 
            _, tm = melting_curve(strand_a = a, strand_b = b, temp_min=5, temp_max=65)
        else:
            if temps[i-1] < 0: 
                temps[i] = temps[i-1]
            else:
                _, tm = melting_curve(strand_a = a, strand_b = b, temp_min=5, temp_max=65)
        temps[i] = tm
    result_arr.append(temps)

canonical_temps = np.zeros(lim)
for i in np.arange(start = 0, stop = lim):
    a_name = f'3C-{i}'
    a = v4_fuel_strand[:-i] if i > 0 else v4_fuel_strand
    b = c3_dna_leg
    _, tm = melting_curve(strand_a = a, strand_b = b)

    canonical_temps[i] = tm

In [ ]:
results = np.vstack(result_arr)
initial_tms = results[:,0].reshape(-1,1)
results_delta = results - initial_tms

canonical_temps_delta = canonical_temps - canonical_temps[0]
print(results_delta)
print(canonical_temps_delta)

random_average = np.mean(results_delta, axis = 0)
print(random_average)

# Generate Tm Figure
fig, ax = plt.subplots(figsize = (10,6))

for i, row in enumerate(results_delta):
    ax.plot(row, color = 'k', alpha = 0.005)
ax.plot(canonical_temps_delta, color = 'r', alpha = 1, label = '3C Fuel')
ax.plot(random_average, color = 'b', alpha = 1, linestyle = '-.', label = 'Mean of Random Strands')
# plt.axhline(y = 22, xmin = 0, xmax = 15, linestyle = '--', color = 'k', label = 'Room Temperature')
ax.set_xlabel('Number of Bases Hydrolyzed', fontsize = 16)
ax.set_ylabel('Change in Melting Temperature (°C)', fontsize = 16)
ax.tick_params(axis = 'both', labelsize =14)
plt.ylim(-55, 5)
plt.legend(fontsize = 14)
plt.tight_layout()
plt.show()